In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

In [3]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

In [4]:
train_data.drop('subject', axis=1, inplace=True)
test_data.drop('subject', axis=1, inplace=True)

In [5]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [6]:
X_train.shape

(5881, 561)

In [7]:
X_test = test_data.drop('Activity', axis=1)
y_test = test_data.loc[:, 'Activity']

In [8]:
y_train.unique()

array(['STANDING', 'LAYING', 'WALKING', 'WALKING_DOWNSTAIRS',
       'WALKING_UPSTAIRS', 'SITTING'], dtype=object)

In [9]:
y_test.unique()

array(['SITTING', 'STANDING', 'WALKING', 'WALKING_UPSTAIRS', 'LAYING',
       'WALKING_DOWNSTAIRS'], dtype=object)

In [10]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

In [11]:
y_test_encoded = le.transform(y_test)

In [12]:
le.classes_

array(['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS',
       'WALKING_UPSTAIRS'], dtype=object)

## 학습 시작

### 1차원 데이터 형태

In [13]:
cnn = Sequential()

In [14]:
cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu',  padding='same', input_shape=(561, 1)))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu',  padding='same'))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 561, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 280, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 280, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 140, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4480)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       573,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 577,574 (2.20 MB)

 Trainable params: 577,574 (2.20 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [17]:
cnn.fit(X_train, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.6563 - loss: 0.8507
Epoch 2/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9262 - loss: 0.1792
Epoch 3/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.9563 - loss: 0.1119
Epoch 4/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9705 - loss: 0.0737
Epoch 5/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.9774 - loss: 0.0646
Epoch 6/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9821 - loss: 0.0523
Epoch 7/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.9871 - loss: 0.0400
Epoch 8/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.9887 - loss: 0.0356
Epoch 9/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 9s 26ms/step - accuracy: 0.9922 - loss: 0.0266
Epoch 10/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.9932 - loss: 0.0236
Epoch 11/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.9940 - loss: 0.0202
Epoch 12/25
184/184 ━━━━━━━━━━━━━━━━━━

In [18]:
y_pred = cnn.predict(X_test)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


In [19]:
y_pred = np.argmax(y_pred, axis=1)

In [20]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.972807613868117
[[291   0   0   0   1   0]
 [  0 221  32   0   0   1]
 [  0   2 284   0   0   1]
 [  0   0   0 228   0   0]
 [  0   0   0   0 193   2]
 [  0   0   0   1   0 214]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       292
           1       0.99      0.87      0.93       254
           2       0.90      0.99      0.94       287
           3       1.00      1.00      1.00       228
           4       0.99      0.99      0.99       195
           5       0.98      1.00      0.99       215

    accuracy                           0.97      1471
   macro avg       0.98      0.97      0.97      1471
weighted avg       0.97      0.97      0.97      1471

Cohen's Kappa Score: 0.9672


## 2차원 데이터 형태

In [21]:
X_train = X_train.values

In [22]:
X_train

array([[ 0.28850798, -0.00919614, -0.10336219, ..., -0.60111976,
         0.33129814,  0.16516332],
       [ 0.26575678, -0.01657558, -0.09816338, ...,  0.34520491,
        -0.76918611, -0.14794449],
       [ 0.27870931, -0.01451131, -0.10871733, ..., -0.83356361,
         0.20243383, -0.03275467],
       ...,
       [ 0.26798135, -0.01834789, -0.10743994, ...,  0.62036305,
        -0.43724704, -0.57183996],
       [ 0.21278703, -0.04813008, -0.12100062, ..., -0.71800894,
         0.29285636,  0.0249202 ],
       [ 0.28381541, -0.04065785, -0.13801804, ..., -0.74024434,
         0.00834582, -0.14614344]])

In [23]:
X_train_reshaped = X_train.reshape(-1, 33, 17, 1)

In [24]:
X_train_reshaped.shape

(5881, 33, 17, 1)

In [25]:
cnn = Sequential()

In [26]:
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(33, 17, 1)))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=6, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 33, 17, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 8, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 8, 32)      │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,542 (552.90 KB)

 Trainable params: 141,542 (552.90 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
cnn.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [29]:
cnn.fit(X_train_reshaped, y_train_encoded, batch_size=32, epochs=25)

Epoch 1/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.5620 - loss: 1.0619
Epoch 2/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.8761 - loss: 0.2909
Epoch 3/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.9219 - loss: 0.1815
Epoch 4/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.9547 - loss: 0.1178
Epoch 5/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.9692 - loss: 0.0845
Epoch 6/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9740 - loss: 0.0691
Epoch 7/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9783 - loss: 0.0594
Epoch 8/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9814 - loss: 0.0487
Epoch 9/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9860 - loss: 0.0383
Epoch 10/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.9896 - loss: 0.0319
Epoch 11/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9915 - loss: 0.0274
Epoch 12/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 

In [30]:
X_test.shape

(1471, 561)

In [31]:
X_test = X_test.values

In [32]:
X_test_reshaped = X_test.reshape(-1, 33, 17, 1)

In [33]:
X_test_reshaped.shape

(1471, 33, 17, 1)

In [34]:
y_pred = cnn.predict(X_test_reshaped)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [35]:
y_pred = np.argmax(y_pred, axis=1)

In [36]:
print('accuracy :',accuracy_score(y_test_encoded, y_pred))
print('='*60)
print(confusion_matrix(y_test_encoded, y_pred))
print('='*60)
print(classification_report(y_test_encoded, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test_encoded, y_pred):.4f}")

accuracy : 0.9809653297076818
[[290   0   0   0   2   0]
 [  0 250   4   0   0   0]
 [  0  17 270   0   0   0]
 [  0   0   0 228   0   0]
 [  0   0   0   0 195   0]
 [  0   0   0   4   1 210]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       292
           1       0.94      0.98      0.96       254
           2       0.99      0.94      0.96       287
           3       0.98      1.00      0.99       228
           4       0.98      1.00      0.99       195
           5       1.00      0.98      0.99       215

    accuracy                           0.98      1471
   macro avg       0.98      0.98      0.98      1471
weighted avg       0.98      0.98      0.98      1471

Cohen's Kappa Score: 0.9771
